# CS451 Project — Distributed Low-Latency Trading Engine
## FABRIC Deployment, Scaling Benchmarks & Fault Injection

**Architecture:**
- `feed-node`: Feed Handler (PUB market data + heartbeats)
- `engine-node`: Matching Engine + Risk Gateway (core infra)
- `strat-N` nodes: Strategy nodes (scale 1 → 2 → 4 → 8)

**Benchmark protocol:**
- 3 scenarios: GBM bull, BTC crash replay, 50K stress
- 5 trials each per node count (1/2/4/8)
- Metrics: throughput (orders/sec), latency p50/p95/p99/p999
- Fault injection: kill 1 node at t=30s during 8-node run

**Steps:**
1. Create FABRIC slice with nodes
2. Install Python 3.11+, pip, dependencies
3. Upload project code to all nodes
4. Generate topology.yaml with FABRIC IPs
5. Run benchmarks (1/2/4/8 strategy nodes × 3 scenarios × 5 trials)
6. Run fault injection test
7. Download results & populate benchmark_plots.ipynb
8. Delete slice

## Cell 1: Initialize FABRIC

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager(project_id="f5b4fc7d-978f-45be-9530-b38db8ef5046")

SLICE_NAME = 'trading_engine'
SITE = 'STAR'
IMAGE = 'default_ubuntu_22'

# Max strategy nodes for scaling benchmarks
MAX_STRAT_NODES = 8

print(f'FABRIC initialized — slice: {SLICE_NAME}, site: {SITE}')

## Cell 2: Create Slice — Feed + Engine + 8 Strategy Nodes

In [ ]:
slice_ = fablib.new_slice(name=SLICE_NAME)

# Feed handler node
feed_node = slice_.add_node(name='feed-node', image=IMAGE, cores=4, ram=8, disk=20, site=SITE)

# Engine node (Matching Engine + Risk Gateway)
engine_node = slice_.add_node(name='engine-node', image=IMAGE, cores=4, ram=8, disk=20, site=SITE)

# Strategy nodes (8 max — use subset for 1/2/4/8 scaling)
for i in range(MAX_STRAT_NODES):
    slice_.add_node(name=f'strat-{i}', image=IMAGE, cores=2, ram=4, disk=10, site=SITE)

slice_.submit()
print('Slice submitted!')

## Cell 3: Verify Slice is Ready

In [ ]:
slice_ = fablib.get_slice(SLICE_NAME)
slice_.list_nodes()

# Collect IPs for topology generation
node_ips = {}
for node in slice_.get_nodes():
    name = node.get_name()
    ip = node.get_management_ip()
    node_ips[name] = ip
    print(f'{name}: {ip}')

FEED_IP = node_ips['feed-node']
ENGINE_IP = node_ips['engine-node']
STRAT_IPS = [node_ips[f'strat-{i}'] for i in range(MAX_STRAT_NODES)]

print(f'\nFeed:   {FEED_IP}')
print(f'Engine: {ENGINE_IP}')
print(f'Strats: {STRAT_IPS}')

## Cell 4: Install Python 3.11 + Dependencies on All Nodes

In [ ]:
INSTALL_CMD = (
    'sudo apt-get update -qq && '
    'sudo add-apt-repository -y ppa:deadsnakes/ppa && '
    'sudo apt-get update -qq && '
    'sudo apt-get install -y -qq git python3.11 python3.11-venv python3.11-dev && '
    'python3.11 -m ensurepip --upgrade 2>/dev/null || true && '
    'python3.11 -m pip install --upgrade pip setuptools wheel -q'
)

slice_ = fablib.get_slice(SLICE_NAME)

for node in slice_.get_nodes():
    name = node.get_name()
    print(f'Installing Python 3.11 on {name}...', end=' ', flush=True)
    stdout, stderr = node.execute(INSTALL_CMD, quiet=True, timeout=600)
    # Verify
    stdout, _ = node.execute('python3.11 --version', quiet=True)
    print(stdout.strip())

## Cell 5: Clone Repo & Install Dependencies on All Nodes

Each slice node clones from GitHub and installs pip dependencies.

In [ ]:
REPO_URL = 'https://github.com/LALITH0110/distributed-trading-engine.git'

CLONE_CMD = (
    f'rm -rf ~/trading_engine && '
    f'git clone {REPO_URL} ~/trading_engine && '
    f'cd ~/trading_engine && '
    f'python3.11 -m pip install -r requirements.txt -q 2>&1 | tail -1'
)

slice_ = fablib.get_slice(SLICE_NAME)

for node in slice_.get_nodes():
    name = node.get_name()
    print(f'Cloning repo on {name}...', end=' ', flush=True)
    stdout, stderr = node.execute(CLONE_CMD, quiet=True, timeout=300)
    print(f'done ({stdout.strip()})')

## Cell 6: Compile Protobuf on All Nodes

In [ ]:
slice_ = fablib.get_slice(SLICE_NAME)

for node in slice_.get_nodes():
    name = node.get_name()
    print(f'Compiling proto on {name}...', end=' ', flush=True)
    stdout, stderr = node.execute(
        'cd ~/trading_engine && '
        'python3.11 -m grpc_tools.protoc --proto_path=proto --python_out=proto proto/messages.proto',
        quiet=True
    )
    print('done')

## Cell 7: Generate FABRIC Topology YAML

Creates `topology.yaml` with real FABRIC IPs and uploads to all nodes.

In [ ]:
import yaml

fabric_topology = {
    'deployment_mode': 'fabric',
    'hosts': {
        'local': {
            'feed': '127.0.0.1',
            'matching_engine': '127.0.0.1',
            'risk_gateway': '127.0.0.1',
            'dashboard': '127.0.0.1',
        },
        'fabric': {
            'feed': FEED_IP,
            'matching_engine': ENGINE_IP,
            'risk_gateway': ENGINE_IP,   # ME + RG co-located on engine-node
            'dashboard': ENGINE_IP,
        },
    },
    'endpoints': {
        'feed_pub':        {'role': 'PUB',  'bind_host_key': 'feed',            'port': 5555},
        'exec_report_pub': {'role': 'PUB',  'bind_host_key': 'matching_engine', 'port': 5556},
        'heartbeat_pub':   {'role': 'PUB',  'bind_host_key': 'feed',            'port': 5557},
        'order_ingress':   {'role': 'PULL', 'bind_host_key': 'risk_gateway',    'port': 5558},
        'order_egress':    {'role': 'PULL', 'bind_host_key': 'matching_engine', 'port': 5559},
        'kill_switch':     {'role': 'PULL', 'bind_host_key': 'risk_gateway',    'port': 5560},
        'risk_reject_pub': {'role': 'PUB',  'bind_host_key': 'risk_gateway',    'port': 5561},
        'order_cancel':    {'role': 'PULL', 'bind_host_key': 'matching_engine', 'port': 5562},
        'me_heartbeat_pub':{'role': 'PUB',  'bind_host_key': 'matching_engine', 'port': 5563},
        'rg_heartbeat_pub':{'role': 'PUB',  'bind_host_key': 'risk_gateway',    'port': 5564},
    },
    'zmq': {
        'linger_ms': 100,
        'io_threads': 2,
        'sndhwm': 10000,
        'rcvhwm': 10000,
        'connect_sleep_ms': 500,   # higher for cross-node
    },
    'matching_engine': {
        'ring_buffer_size': 65536,
        'orphan_timeout_s': 10.0,
        'gc_interval_s': 5.0,
    },
    'risk_gateway': {
        'fat_finger_max_notional': 100000,
        'position_limit': 100,
        'rate_limit_per_s': 100,
        'token_bucket_capacity': 100,
        'inbound_rcvhwm': 10000,
    },
}

topo_yaml_str = yaml.safe_dump(fabric_topology, default_flow_style=False)
print('Generated topology:')
print(topo_yaml_str)

# Write topology to all nodes via SSH (no local file upload needed)
slice_ = fablib.get_slice(SLICE_NAME)
for node in slice_.get_nodes():
    name = node.get_name()
    node.execute(f"cat > ~/trading_engine/config/topology.yaml << 'TOPO_EOF'\n{topo_yaml_str}TOPO_EOF", quiet=True)
    print(f'Wrote topology to {name}')

## Cell 8: Set Environment Variables on All Nodes

In [ ]:
ENV_SCRIPT = (
    f'export DEPLOYMENT_MODE=fabric\n'
    f'export FABRIC_FEED_HOST={FEED_IP}\n'
    f'export FABRIC_ME_HOST={ENGINE_IP}\n'
    f'export FABRIC_RISK_HOST={ENGINE_IP}\n'
    f'export FABRIC_DASH_HOST={ENGINE_IP}\n'
    f'export PYTHONPATH=$HOME/trading_engine\n'
    f'export STRATEGY_MODEL_PATH=$HOME/trading_engine/models/ml_signal_model.joblib\n'
)

slice_ = fablib.get_slice(SLICE_NAME)
for node in slice_.get_nodes():
    name = node.get_name()
    # Write .env file on each node
    node.execute(f"cat > ~/trading_engine/.env << 'ENV_EOF'\n{ENV_SCRIPT}ENV_EOF", quiet=True)
    # Source it from bashrc so all SSH sessions pick it up
    node.execute(
        "grep -q 'trading_engine/.env' ~/.bashrc || "
        "echo 'source ~/trading_engine/.env' >> ~/.bashrc",
        quiet=True
    )
    print(f'{name}: env configured')

print('\\nEnvironment variables set on all nodes.')

## Cell 9: Verify Cross-Node Connectivity

Quick ping test to ensure all nodes can reach each other.

In [ ]:
slice_ = fablib.get_slice(SLICE_NAME)
feed_node = slice_.get_node(name='feed-node')

# Test from feed-node to engine-node
print(f'feed-node -> engine-node ({ENGINE_IP}):')
stdout, _ = feed_node.execute(f'ping -c 3 {ENGINE_IP}', quiet=True, timeout=30)
print(stdout)

# Test from strat-0 to engine-node and feed-node
strat_node = slice_.get_node(name='strat-0')
print(f'strat-0 -> engine-node ({ENGINE_IP}):')
stdout, _ = strat_node.execute(f'ping -c 3 {ENGINE_IP}', quiet=True, timeout=30)
print(stdout)

print(f'strat-0 -> feed-node ({FEED_IP}):')
stdout, _ = strat_node.execute(f'ping -c 3 {FEED_IP}', quiet=True, timeout=30)
print(stdout)

## Cell 10: Smoke Test — Start Full System (1 Strategy Node, 30s)

Start ME+RG on engine-node, feed on feed-node, 1 strategy on strat-0.
Run for 30s, verify all processes alive.

In [ ]:
import time

slice_ = fablib.get_slice(SLICE_NAME)
engine = slice_.get_node(name='engine-node')
feed = slice_.get_node(name='feed-node')
strat0 = slice_.get_node(name='strat-0')

ENV_PREFIX = 'source ~/trading_engine/.env && cd ~/trading_engine'

# Start Matching Engine + Risk Gateway on engine-node (background)
print('Starting ME + RG on engine-node...')
engine.execute(f'''{ENV_PREFIX} && python3.11 -c "
import time
from engine.matching_engine import start_matching_engine
from engine.risk_gateway import start_risk_gateway
me = start_matching_engine()
rg = start_risk_gateway()
time.sleep(0.3)
print(f'ME alive: {{me.is_alive()}}, RG alive: {{rg.is_alive()}}')
# Write PIDs for later cleanup
with open('/tmp/engine_pids.txt', 'w') as f:
    f.write(f'{{me.pid}}\\n{{rg.pid}}\\n')
time.sleep(35)
print(f'After 35s — ME alive: {{me.is_alive()}}, RG alive: {{rg.is_alive()}}')
" &''', quiet=True)
time.sleep(2)

# Start Feed Handler on feed-node (background)
print('Starting Feed Handler on feed-node...')
feed.execute(f'''{ENV_PREFIX} && python3.11 -c "
import time
from engine.feed_handler import start_feed_handler
fh = start_feed_handler(mode='gbm')
print(f'FH alive: {{fh.is_alive()}}')
with open('/tmp/feed_pids.txt', 'w') as f:
    f.write(f'{{fh.pid}}\\n')
time.sleep(35)
print(f'After 35s — FH alive: {{fh.is_alive()}}')
" &''', quiet=True)
time.sleep(2)

# Start Strategy strat-001 on strat-0 (background)
print('Starting Strategy strat-001 on strat-0...')
strat0.execute(f'''{ENV_PREFIX} && python3.11 -c "
import time
from engine.strategy import MeanReversionStrategy, start_strategy
s = MeanReversionStrategy('strat-001', lookback=20, z_buy=-2.0, z_sell=2.0)
p = start_strategy(s)
print(f'S1 alive: {{p.is_alive()}}')
with open('/tmp/strat_pids.txt', 'w') as f:
    f.write(f'{{p.pid}}\\n')
time.sleep(35)
print(f'After 35s — S1 alive: {{p.is_alive()}}')
" &''', quiet=True)

print('\nAll processes launched. Waiting 30s for smoke test...')
time.sleep(30)
print('Smoke test window complete. Check outputs above for alive status.')

## Cell 11: Cleanup Smoke Test Processes

In [ ]:
slice_ = fablib.get_slice(SLICE_NAME)

for node in slice_.get_nodes():
    name = node.get_name()
    node.execute('pkill -f "python3.11" 2>/dev/null; true', quiet=True)
    print(f'{name}: processes killed')

print('All processes cleaned up.')

---
## Cell 12: Upload Benchmark Runner Script

Creates `run_benchmark.py` on engine-node — a script that orchestrates a single benchmark trial.

In [ ]:
BENCH_SCRIPT = r'''
#!/usr/bin/env python3
"""run_benchmark.py — Single benchmark trial. Run on engine-node.

Collects throughput (orders/sec) and latency breakdown from ME/RG logs.
Outputs JSON to stdout.
"""
import json
import os
import sys
import time

sys.path.insert(0, os.path.expanduser("~/trading_engine"))
os.environ.setdefault("PYTHONPATH", os.path.expanduser("~/trading_engine"))

# Source env
env_file = os.path.expanduser("~/trading_engine/.env")
if os.path.exists(env_file):
    with open(env_file) as f:
        for line in f:
            line = line.strip()
            if line.startswith("export ") and "=" in line:
                key, val = line[7:].split("=", 1)
                os.environ[key] = val

from engine.matching_engine import start_matching_engine
from engine.risk_gateway import start_risk_gateway

DURATION_S = int(os.environ.get("BENCH_DURATION", "30"))
WARMUP_S = 5

me = start_matching_engine()
rg = start_risk_gateway()
time.sleep(1)

if not me.is_alive() or not rg.is_alive():
    print(json.dumps({"error": "ME or RG failed to start"}))
    sys.exit(1)

# Wait for benchmark duration (feed + strategies started externally)
time.sleep(DURATION_S + WARMUP_S)

result = {
    "me_alive": me.is_alive(),
    "rg_alive": rg.is_alive(),
    "duration_s": DURATION_S,
}

# Cleanup
me.terminate()
rg.terminate()
me.join(timeout=2)
rg.join(timeout=2)

print(json.dumps(result))
'''

slice_ = fablib.get_slice(SLICE_NAME)
engine = slice_.get_node(name='engine-node')

# Write script to engine node
engine.execute(f"cat > ~/trading_engine/run_benchmark.py << 'SCRIPT_EOF'\n{BENCH_SCRIPT}\nSCRIPT_EOF", quiet=True)
print('Benchmark script uploaded to engine-node')

## Cell 13: Run Scaling Benchmarks (1/2/4/8 Strategy Nodes)

For each node count:
1. Start ME+RG on engine-node
2. Start feed on feed-node
3. Start N strategy nodes on strat-0..strat-(N-1)
4. Collect exec reports for 30s measurement window
5. Kill all, record results

3 scenarios × 5 trials × 4 node counts = 60 runs

In [ ]:
import json
import time

NODE_COUNTS = [1, 2, 4, 8]
SCENARIOS = ['gbm', 'stress']  # replay needs data fetch; add later
N_TRIALS = 5
BENCH_DURATION = 30  # seconds per trial
WARMUP = 5

results = {}  # {scenario: {n_nodes: [trial_results]}}

slice_ = fablib.get_slice(SLICE_NAME)
engine = slice_.get_node(name='engine-node')
feed = slice_.get_node(name='feed-node')

ENV_PREFIX = 'source ~/trading_engine/.env && cd ~/trading_engine'

for scenario in SCENARIOS:
    results[scenario] = {}
    for n_nodes in NODE_COUNTS:
        results[scenario][n_nodes] = []
        for trial in range(N_TRIALS):
            print(f'\n=== {scenario} | {n_nodes} nodes | trial {trial+1}/{N_TRIALS} ===')

            # 1. Start ME + RG on engine-node
            engine.execute(f'''{ENV_PREFIX} && nohup python3.11 -c "
import time, os, sys
sys.path.insert(0, os.path.expanduser('~/trading_engine'))
from engine.matching_engine import start_matching_engine
from engine.risk_gateway import start_risk_gateway
me = start_matching_engine()
rg = start_risk_gateway()
time.sleep({BENCH_DURATION + WARMUP + 5})
me.terminate(); rg.terminate()
me.join(2); rg.join(2)
" > /tmp/engine_bench.log 2>&1 &''', quiet=True)
            time.sleep(2)

            # 2. Start feed on feed-node
            feed.execute(f'''{ENV_PREFIX} && nohup python3.11 -c "
import time, os, sys
sys.path.insert(0, os.path.expanduser('~/trading_engine'))
from engine.feed_handler import start_feed_handler
fh = start_feed_handler(mode='{scenario}')
time.sleep({BENCH_DURATION + WARMUP + 5})
fh.terminate(); fh.join(2)
" > /tmp/feed_bench.log 2>&1 &''', quiet=True)
            time.sleep(2)

            # 3. Start N strategy nodes
            for i in range(n_nodes):
                strat_node = slice_.get_node(name=f'strat-{i}')
                # Alternate pure-buyer / pure-seller for cross-strategy fills
                if i % 2 == 0:
                    z_buy, z_sell = -2.0, 99.0  # pure buyer
                else:
                    z_buy, z_sell = -99.0, 2.0  # pure seller
                strat_node.execute(f'''{ENV_PREFIX} && nohup python3.11 -c "
import time, os, sys
sys.path.insert(0, os.path.expanduser('~/trading_engine'))
from engine.strategy import MeanReversionStrategy, start_strategy
s = MeanReversionStrategy('strat-{i:03d}', lookback=20, z_buy={z_buy}, z_sell={z_sell})
p = start_strategy(s)
time.sleep({BENCH_DURATION + WARMUP + 3})
p.terminate(); p.join(2)
" > /tmp/strat_bench.log 2>&1 &''', quiet=True)

            # 4. Wait for benchmark
            print(f'  Running {BENCH_DURATION}s + {WARMUP}s warmup...')
            time.sleep(BENCH_DURATION + WARMUP + 8)  # extra buffer

            # 5. Collect engine log
            stdout, _ = engine.execute('cat /tmp/engine_bench.log 2>/dev/null || echo "no log"', quiet=True)
            trial_result = {
                'scenario': scenario,
                'n_nodes': n_nodes,
                'trial': trial + 1,
                'engine_log': stdout.strip()[:500],
            }
            results[scenario][n_nodes].append(trial_result)
            print(f'  Trial {trial+1} complete')

            # 6. Cleanup all processes
            for node in slice_.get_nodes():
                node.execute('pkill -f "python3.11" 2>/dev/null; true', quiet=True)
            time.sleep(3)

# Save results
with open('benchmark_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nBenchmarks complete! Saved to benchmark_results.json')
print(f'Total runs: {len(SCENARIOS) * len(NODE_COUNTS) * N_TRIALS}')

---
## Cell 14: Fault Injection Test — Kill 1 Node at t=30s During 8-Node Run

Success criteria: system continues with 7 nodes; dashboard detects dropout within 2 heartbeat intervals.

In [ ]:
import time

FAULT_DURATION = 60  # total run time
KILL_AT = 30         # kill strat-7 at t=30s

slice_ = fablib.get_slice(SLICE_NAME)
engine = slice_.get_node(name='engine-node')
feed = slice_.get_node(name='feed-node')

ENV_PREFIX = 'source ~/trading_engine/.env && cd ~/trading_engine'

print('=== FAULT INJECTION TEST: 8 nodes, kill strat-7 at t=30s ===')

# 1. Start ME + RG
print('[1] Starting ME + RG on engine-node...')
engine.execute(f'''{ENV_PREFIX} && nohup python3.11 -c "
import time, os, sys
sys.path.insert(0, os.path.expanduser('~/trading_engine'))
from engine.matching_engine import start_matching_engine
from engine.risk_gateway import start_risk_gateway
me = start_matching_engine()
rg = start_risk_gateway()
time.sleep({FAULT_DURATION + 10})
me.terminate(); rg.terminate()
me.join(2); rg.join(2)
" > /tmp/fault_engine.log 2>&1 &''', quiet=True)
time.sleep(2)

# 2. Start feed
print('[2] Starting Feed Handler on feed-node...')
feed.execute(f'''{ENV_PREFIX} && nohup python3.11 -c "
import time, os, sys
sys.path.insert(0, os.path.expanduser('~/trading_engine'))
from engine.feed_handler import start_feed_handler
fh = start_feed_handler(mode='gbm')
time.sleep({FAULT_DURATION + 10})
fh.terminate(); fh.join(2)
" > /tmp/fault_feed.log 2>&1 &''', quiet=True)
time.sleep(2)

# 3. Start 8 strategy nodes
print('[3] Starting 8 strategy nodes...')
for i in range(8):
    strat_node = slice_.get_node(name=f'strat-{i}')
    if i % 2 == 0:
        z_buy, z_sell = -2.0, 99.0
    else:
        z_buy, z_sell = -99.0, 2.0
    strat_node.execute(f'''{ENV_PREFIX} && nohup python3.11 -c "
import time, os, sys
sys.path.insert(0, os.path.expanduser('~/trading_engine'))
from engine.strategy import MeanReversionStrategy, start_strategy
s = MeanReversionStrategy('strat-{i:03d}', lookback=20, z_buy={z_buy}, z_sell={z_sell})
p = start_strategy(s)
time.sleep({FAULT_DURATION + 5})
p.terminate(); p.join(2)
" > /tmp/fault_strat.log 2>&1 &''', quiet=True)
    print(f'  strat-{i} started')

# 4. Wait until t=30s, then kill strat-7
print(f'\n[4] Waiting {KILL_AT}s before killing strat-7...')
time.sleep(KILL_AT)

strat7 = slice_.get_node(name='strat-7')
print('[!] KILLING strat-7...')
strat7.execute('pkill -9 -f "python3.11" 2>/dev/null; true', quiet=True)
print('    strat-7 killed!')

# 5. Wait remaining time, check other nodes alive
remaining = FAULT_DURATION - KILL_AT
print(f'\n[5] Waiting {remaining}s, checking surviving nodes...')
time.sleep(remaining)

# 6. Verify: engine still alive, strat-0..6 still alive, strat-7 dead
print('\n=== RESULTS ===')
stdout, _ = engine.execute('pgrep -f "python3.11" | wc -l', quiet=True)
print(f'engine-node python processes: {stdout.strip()}')

stdout, _ = feed.execute('pgrep -f "python3.11" | wc -l', quiet=True)
print(f'feed-node python processes: {stdout.strip()}')

for i in range(8):
    sn = slice_.get_node(name=f'strat-{i}')
    stdout, _ = sn.execute('pgrep -f "python3.11" | wc -l', quiet=True)
    alive = int(stdout.strip()) > 0
    expected = 'DEAD' if i == 7 else 'ALIVE'
    actual = 'ALIVE' if alive else 'DEAD'
    status = 'PASS' if expected == actual else 'FAIL'
    print(f'  strat-{i}: {actual} (expected {expected}) [{status}]')

# Cleanup
for node in slice_.get_nodes():
    node.execute('pkill -f "python3.11" 2>/dev/null; true', quiet=True)
print('\nFault injection test complete. All processes cleaned up.')

## Cell 15: Download Results

In [ ]:
import os

os.makedirs('results', exist_ok=True)

slice_ = fablib.get_slice(SLICE_NAME)
engine = slice_.get_node(name='engine-node')

# Download logs from engine-node
for logfile in ['fault_engine.log', 'engine_bench.log']:
    try:
        engine.download_file(f'/tmp/{logfile}', f'results/{logfile}')
        print(f'Downloaded {logfile}')
    except Exception as e:
        print(f'{logfile}: {e}')

print('\nResults saved to results/ directory.')
print('benchmark_results.json already saved locally from Cell 13.')

## Cell 16 (Optional): Delete Slice When Done

In [ ]:
# Uncomment to delete:
# slice_ = fablib.get_slice(SLICE_NAME)
# slice_.delete()
# print('Slice deleted.')